In [2]:
import sys
sys.path.append("../")

In [8]:
import openai
import src.process as conspectum

with open("secret/openai_key.txt", "r", encoding = "utf-8") as file:
    openai_key = file.read()
with open("secret/openai_base_url.txt", "r", encoding = "utf-8") as file:
    openai_base_url = file.read()

ai = openai.AsyncOpenAI(api_key=openai_key, base_url=openai_base_url)


In [ ]:

with open("./output_cut_big.wav", "rb") as audio_file:
    audio = await conspectum.preprocess(audio_file.read(), ai, verbose_dir = "out")

In [3]:
audio

'Title: Introduction to Information Theory and Shannon Entropy\n\n137\n322\n497\n709\n'

In [17]:
import pydub
import io
with open("./output_cut_big.wav", "rb") as audio_file:
    a = pydub.AudioSegment.from_file(io.BytesIO(audio_file.read()), format="wav")
len(a)

600000

In [ ]:
from pydub import AudioSegment
import numpy as np
import matplotlib.pyplot as plt

def analyze_audio_dbfs(audio_path, window_ms=100):
    # Load audio
    audio = AudioSegment.from_file(audio_path)
    samples = np.array(audio.set_channels(1).get_array_of_samples())
    samples = samples.astype(np.float32) / (2**15)  # Normalize to [-1, 1]

    # Frame rate and window size
    frame_rate = audio.frame_rate
    window_samples = int(frame_rate * (window_ms / 1000))

    dbfs_values = []

    # Slide window and compute RMS → dBFS
    for i in range(0, len(samples), window_samples):
        frame = samples[i:i + window_samples]
        if len(frame) == 0:
            continue
        rms = np.sqrt(np.mean(frame**2))
        if rms > 0:
            dbfs = 20 * np.log10(rms)
        else:
            dbfs = -float('inf')
        dbfs_values.append(dbfs)

    return np.array(dbfs_values)

dbfs_data = analyze_audio_dbfs("output_cut_big.wav", window_ms=2000)

print(f"Min dBFS: {np.min(dbfs_data):.1f} dB")
print(f"Max dBFS: {np.max(dbfs_data):.1f} dB")
print(f"Mean dBFS: {np.mean(dbfs_data):.1f} dB")
print(f"Median dBFS: {np.median(dbfs_data):.1f} dB")

plt.hist(dbfs_data, bins=50, range=(-80, 0), alpha=0.7, color='blue')
plt.xlabel('dBFS')
plt.ylabel('Count')
plt.title('Distribution of Audio Levels (RMS in dBFS)')
plt.grid(True)
plt.show()

np.sort(dbfs_data)[:10]

In [ ]:
vol = analyze_audio_dbfs("output_cut_big.wav", window_ms=1)
vol

In [ ]:
t = []
for i in range(len(vol)):
    t.append(vol[i:i+2000].max())
np.sort(t)

In [3]:
from pydub import AudioSegment
from pydub.silence import split_on_silence

audio = AudioSegment.from_file("output_cut_big.wav")

chunks = split_on_silence(
    audio,
    min_silence_len=2000,      # 2 seconds
    silence_thresh=-51,       # dBFS
    keep_silence=500           # Keep 0.5s of silence at edges
)


In [4]:
[len(chunk) for chunk in chunks]

[70955, 130619, 118851, 46389, 3579, 116482, 103890]

In [ ]:
from tmp import process_to_tex

from pydub import AudioSegment
from pydub.silence import split_on_silence
from io import BytesIO

buffer = BytesIO()
chunks[0].export(buffer, format="wav")
wav_bytes = buffer.getvalue()

tex = await process_to_tex(wav_bytes, "wav", ai)
tex

In [ ]:
import pdflatex
from tmp import postprocess_tex

post_tex = postprocess_tex(tex)
with open("chunk0.tex", "w") as file:
    file.write(tex)

import uuid
pdf = pdflatex.PDFLaTeX(post_tex, uuid.uuid4()).create_pdf()[0]

with open("chunk0.pdf", "wb") as file:
    file.write(pdf)

In [29]:
with open("../src/prompts/summary_prompt.txt") as prompt_file:
    summary_prompt = prompt_file.read()
summary_prompt

'You are an academic summarizer. Your task is to listen to an academic lecture and produce:\n1. A concise, informative title (10–15 words max) that captures the core topic and focus of the lecture.\n2. A short abstract (80–120 words) that summarizes:\n  - The main topic and learning objectives,\n  - Key concepts or theories discussed,\n  - Any significant examples, methods, or conclusions presented.\n\nGuidelines:\n- Use clear, academic language appropriate for scholarly contexts.\n- Avoid jargon unless essential; explain briefly if needed.\n- Do not include personal opinions or extraneous details.\n- Ensure the abstract is self-contained: someone reading it should understand the lecture’s essence without seeing the full transcript.\n\nOutput format:\n- Put you generated title at the first line of the output.\n- Then, followed by an empty line, place the generated concise summary.'

In [30]:
import base64
with open("./output_cut_big.wav", "rb") as audio_file:
    base64_encoded = base64.b64encode(audio_file.read()).decode("utf-8")
len(base64_encoded)

76800104

In [31]:
response = await ai.chat.completions.create(
    model="gpt-4o-audio-preview",
    modalities=["text"],
    messages=[
        {"role": "system", "content": summary_prompt},
        {
            "role": "user",
            "content":
            [
                {"type": "input_audio", "input_audio": {"data": base64_encoded, "format": "wav"}}
            ],
        },
    ],
)

In [32]:
response.choices[0].message.content

'The Concept of Information and Entropy in Probability Theory: From Shannon’s Definition to System Uncertainty\n\nThis lecture introduces and elaborates on the foundational concepts of information theory as developed by Claude Shannon, focusing on the relationship between probability, information, and entropy. The core objective is to understand how information is quantitatively measured in probabilistic systems and how entropy serves as a metric for system uncertainty or unpredictability. The speaker begins by contrasting two probabilistic events: a professor entering a room, and a professor attacking a student. Since the latter is rarer, it carries more informational value when it occurs—highlighting that the more improbable an event, the more information it conveys. Shannon defined information as the logarithm of the inverse probability of an event.\n\nThe lecture further explains entropy as the average information of all possible system states, weighted by their probabilities. This

In [43]:
from pydub import AudioSegment

audio = AudioSegment.from_file("output_cut_big.wav")

chunks = [
    audio[0   * 1000:123 * 1000],
    audio[123 * 1000:296 * 1000],
    audio[296 * 1000:489 * 1000],
    audio[489 * 1000:]
]
[len(chunk) for chunk in chunks]

[123000, 173000, 193000, 111000]

In [5]:
with open("../src/prompts/system_prompt.txt") as prompt_file:
    system_prompt = prompt_file.read()
system_prompt

'You are an expert academic assistant specialized in transforming spoken lecture content into beautifully formatted, publication-ready LaTeX documents. Your role is to process lecture audio and produce a high-quality summary that is: accurate, structured, mathematically precise, and visually appealing.\n\nYou will be given a template to follow. This template contains additional instructions to help understand the document formatting rules. Please also adhere to these guidelines. I will now briefly outline the primary requirements:\n- Use \\section{}, \\subsection{}, etc., to reflect lecture flow.\n- Use different fonts and text styling to highlight information.\n- Convert spoken math into proper LaTeX math mode ($...$, $$...$$, \\begin{equation}...\\end{equation}).\n- Highlight definitions, theorems, etc. using thmbox, defbox, lembox, propbox, corbox, exbox, and rembox.\n- Use itemized or enumerated lists for steps, examples, or key points.\n- Create footnotes with a glossary of terms 

In [6]:
with open("../src/prompts/template.tex") as prompt_file:
    template = prompt_file.read()
template

'\\documentclass[12pt,a4paper]{article}\n\n\\usepackage[utf8]{inputenc}\n\\usepackage[T1]{fontenc}\n\\usepackage{lmodern}\n\\usepackage[english]{babel}\n\\usepackage{amsmath, amssymb, amsthm}\n\\usepackage{tcolorbox}\n\\usepackage{enumitem}\n\\usepackage{array, booktabs}\n\\usepackage{multicol}\n\\usepackage[a4paper, margin=1in]{geometry}\n\\usepackage{fancyhdr}\n\\usepackage{hyperref}\n\\usepackage{cleveref}\n\\usepackage{graphicx}\n\\usepackage{caption}\n\\usepackage{footmisc}\n\\usepackage{xcolor}\n\n\\geometry{top=2.5cm, bottom=2.5cm, left=2.5cm, right=2.5cm, headheight=15pt}\n\\hypersetup{colorlinks=true, linkcolor=blue, filecolor=magenta, urlcolor=cyan, pdfpagemode=FullScreen}\n\n\\pagestyle{fancy}\n\\fancyhf{}\n\\lhead{\\footnotesize \\textit{Lecture Summary}}\n\\rhead{\\thepage}\n\n% Custom colors for tcolorbox\n\\tcbuselibrary{theorems, skins, breakable}\n\\definecolor{thmcolor}{RGB}{200,230,255}\n\\definecolor{defcolor}{RGB}{230,250,220}\n\\definecolor{excolor}{RGB}{255,240,2

In [23]:
from io import BytesIO
import base64

raw_results = []
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i + 1}/{len(chunks)}")
    buffer = BytesIO()
    chunk.export(buffer, format="wav")
    wav_bytes = buffer.getvalue()

    base64_encoded = base64.b64encode(wav_bytes).decode("utf-8")
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": f"The template:\n\n{template}"},
        {
            "role": "user",
            "content":
            [
                {"type": "text", "text": f"This is chunk {i + 1}/{len(chunks)} of the lecture."},
                {"type": "input_audio", "input_audio": {"data": base64_encoded, "format": "wav"}}
            ],
        },
    ]
    if len(raw_results) != 0:
        last_section_start = max(raw_results[-1].rfind('\\section'), raw_results[-1].rfind('\\subsection'))
        messages.append(
            {"role": "system", "content": f"Previous chunk finished with the following:\n\n{raw_results[-1][last_section_start:]}"}
        )

    with open("t.txt", "w") as file:
        file.write(str(messages))
    response = await ai.chat.completions.create(
        model="gpt-4o-audio-preview",
        modalities=["text"],
        messages=messages
    )
    raw_results.append(response.choices[0].message.content)

Processing chunk 1/7
Processing chunk 2/7
Processing chunk 3/7
Processing chunk 4/7
Processing chunk 5/7
Processing chunk 6/7
Processing chunk 7/7


In [24]:
raw_results

['\\section{Introduction to Probabilistic Events and Information Gain}\nThis section begins by presenting a classical example in the domain of probabilistic events and their informational content, as proposed by German scientist Hermann Haken, a pioneer in the field of Synergetics. The discussion revolves around comparing two distinct probabilistic events and evaluating which of them brings a greater amount of information.\n\n\\subsection{Setting up the Example}\nWe define the following two probabilistic events:\n\\begin{itemize}\n    \\item \\textbf{Event A}: A professor, referred to as Professor N, enters the classroom.\n    \\item \\textbf{Event B}: The same professor, Professor N, accuses a student.\n\\end{itemize}\n\nThe central question posed in this lecture is: \\emph{Which of these probabilistic events carries a greater amount of information?}\n\n\\subsection{Introduction to Information and Probability}\nThe core idea here relates to the relationship between information content

In [25]:
last_section_start = max(raw_results[0].rfind('\\section'), raw_results[0].rfind('\\subsection'))
raw_results[0][last_section_start:]

'\\subsection{Introduction to Information and Probability}\nThe core idea here relates to the relationship between information content and the likelihood of an event. Generally, the more improbable an event is, the more information it carries when it occurs.\n\nTo understand this concept, we consider the idea of \\textbf{surprise} or \\textbf{information gain}, which is tied to how unexpected an event is:\n\\begin{itemize}\n    \\item Common events (high probability) bring less information.\n    \\item Rare events (low probability) bring more information.\n\\end{itemize}\n\n\\begin{defbox}[Information Content of an Event]\nThe information content (sometimes referred to as self-information) of an event $E_i$ with probability $P(E_i)$ is defined as:\n\\[\nI(E_i) = -\\log_2 P(E_i)\n\\]\nwhere $P(E_i)$ is the probability of the event occurring. The less likely the event, the higher its information content.\n\\end{defbox}\n\nIn the given example, the goal is to judge whether Event A (profes

In [26]:
result = template.replace("%% <INSERT CONTENT HERE>", '\n'.join(raw_results))
with open("result.tex", "w") as out:
    out.write(result)

In [ ]:
import tqdm
import pydub
import pydub.utils

thr = []
for i in tqdm.trange(len(audio)):
    thr.append(pydub.utils.ratio_to_db(audio[i:i+2000].rms / audio.max_possible_amplitude))
thr

In [ ]:
np.sort(thr)[:10]

In [ ]:
pydub.utils.db_to_float(-51) # * audio_segment.max_possible_amplitude

In [ ]:
pydub.utils.ratio_to_db(0.002818382931264455)

In [ ]:
chunks = 

In [ ]:
len(audio)

In [ ]:
sum([len(chunk) for chunk in chunks])

In [ ]:
[len(chunk) for chunk in chunks]

In [ ]:
len(chunks)

In [ ]:
for i, chunk in enumerate(chunks):
    chunk.export(f"{i}.wav")

In [ ]:
chunks[0].export("test.wav")

In [ ]:
import ffmpeg

res = (
    ffmpeg.FFmpeg()
    .input("output_cut_big.wav")
)
res


# ffmpeg -i output_cut_big.wav -af "silencedetect=noise=-40dB:d=1" -f null - -loglevel info

        .input('output_cut_big.wav')
        .filter('silencedetect', noise='-40dB', d=1)
        .output('pipe:', format='null')  # equivalent to -f null -
        .overwrite_output()
        .run(capture_stdout=True, capture_stderr=True)

In [ ]:
from ffmpeg import FFmpeg


def main():
    ffmpeg = (
        FFmpeg()
        .option("y")
        .input("input.mp4")
        .output(
            "output.mp4",
            {"codec:v": "libx264"},
            vf="scale=1280:-1",
            preset="veryslow",
            crf=24,
        )
    )

    ffmpeg.execute()